---
authors:
  - edesz
date: 2025-10-04
---

# Validate Raw Data

In [ ]:
import os
import sys
from pathlib import Path

import boto3
import botocore.exceptions
import numpy as np
import pandas as pd
import pandera as pa
from dotenv import load_dotenv
from pandera import Check, Field
from pandera.typing import Series
from scipy import stats

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import r2.io_utils as r2io
from utils.df_utils import show_df

In [ ]:
data_models_path = str(PROJ_ROOT)
if data_models_path not in sys.path:
    sys.path.append(data_models_path)

In [ ]:
from data_models.staging import CreditCardCustomerSchema

## About

Having completed the task of identifying the cohort of at-risk customers that maximizes Return on Investment (ROI), we now need to ensure new customer data has the same characteristics as the sample of data that was provided to us by the client. This is necessary to ensure we our understanding of the bank's new (unseen) credit card customers' characteristics has not changed. By doing this, we ensure the analysis we developed using the historical data can also be applied to the new customers data and the client can realize the estimated ROI from the best cohort of at-risk customers. Any *data drift* can lead the client targeting the wrong customers or miss those truly at risk, which would waste marketing resources and not achieve our estimated ROI.

With this in mind, in this notebook, we develop a customer schema to validate raw credit card customer data. The schema is used to validate the raw historical data that the client gave us for which the outcome of churn is known. The same schema can be used to validate new data as well, with no changes.

[Pandera](https://pypi.org/project/pandera/) provides a robust framework to implement this validation. It allows for defining a formal schema that checks for correct data types, value ranges, and specific categories for every customer attribute. By using Pandera, we can automatically catch the customers that do not have the same raw characteristics of the customers in the historical data. It can also check for missing values since we did not have any in the historical customer data.

In summary, this step ensures the new credit card dataset has the same characteristics of the historical sample of data that we used to identify at-risk customers and to estimate cohort size and predicted ROI.

### Output

None

## User Inputs

In [ ]:
# name of raw data key (file) in private R2 bucket
r2_key_raw_data = "BankChurners.xlsx"

In [ ]:
account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Extract

In [ ]:
%%time
df = r2io.pandas_read_xlsx_r2(s3_client, bucket_name, r2_key_raw_data, {})
print(f"Loaded {len(df):,} rows of raw data")
_ = show_df(df)
with pd.option_context('display.max_columns', None):
    display(df.head(1))

## Data Validatation

In this section data validation checks performed on all columns in the raw data using the [`pandera` Python package](https://pypi.org/project/pandera/).

Three types of checks are performed

1. at the **column level** on individual columns
   - for numerical columns
     - to ensure their datatype and bounds (minimum and/or maximum values) are expected from the sample data
     - to ensure their Z-scores are expected from the sample data
   - for categorical and ordinal columns
     - to ensure their unique values are the same as those from the sample data
   - for the customer identifier column
     - to ensure the identifier is unique to each customer
2. at the **dataset level** to ensure inter-column relationships are in line with those expected from the sample data
3. at the **observation (row) level** on individual customers based on their credit card activity

### Checks

#### Column-Level Checks - Part 1/2 (Datatypes, Bounds, Expected Values)

**Categorical and Ordinal Columns and Class Label (expected values using `isin`)**

1. Ensures values belong to a predefined set
2. Prevents unexpected categories due to data drift or ingestion errors

**Numerical Columns (bounds using `ge`, `le`, `gt`)**

1. Enforce realistic domain limits (e.g., age, months, ratios)
2. Prevent invalid or corrupted values (e.g., negative balances)

**Identifier Column `CLIENTNUM` (uniqueness)**

1. Ensures each customer is uniquely identified
2. Prevents duplicate records

#### Dataset-Level Checks

Next, we focus on the relationships between columns in the raw dataset. We expect the following relationships

**Credit Balance**

Credit_Limit ~ Total_Revolving_Bal + Avg_Open_To_Buy

**Utilization Ratio**

Avg_Utilization_Ratio ~ Total_Revolving_Bal / Credit_Limit

#### Anomaly Detection Based on Credit Card Customer Behaviour

Next, based on the data provided to us by the client, we do not expect customers with the following criteria in the dataset

1. High utilization (≥ 0.8) and low activity
2. Transaction count ≤ 10
3. Transaction amount ≤ 1000

Any such customers should be classified as an anomaly. As an example, we want to detect if customers are present with  a maxed-out their credit limit but with little credit card usage, which is an unrealistic scenario. So a validation check is implemented to capture such risky or unusual patterns in customer behavior.

### Column-Level Checks - Part 2/2 (Z-Score to Detect Outliers)

Finally, we use z-score tests to check for outliers in numerical columns.

Based on the sample data provided to us by the client, we expect the following

**|Z-Score| ≤ 4.0**

The most common z-score threshold is 3 (or -3), meaning data points (rows) further than 3 standard deviations from the mean will fail this test. Based on our EDA of the sample of the data provided by the client, we have outliers in the sample data, so we are allowing extreme outliers by accepting a z-score of up to 4.

These checks identify extreme values across the numerical columns, which helps detect

1. data entry errors
2. rare but suspicious customer records
3. anomalies in the numerical variable's distribution

A row (customer) fails this test if any monitored column is an outlier. We are using the population standard deviation, which uses a `ddof=0`, which handles zero-variance columns safely.

### Property-Based Tests

`pandera` has `Hypothesis` testing which allows for running statistical tests like `two_sample_ttest` to validate data relationships by checking for distributions or differences between data groups. This allows us to define tests beyond simple value checks. Here, seven such tests were used as part of the data schema.

### Implementation

Based on the above requirements, the schema is split into three files in `data_models/staging` to keep the Business Logic (Hypotheses) separate from the Data Integrity (Types/Ranges)

1. `base_schema.py`
   - this contains the `pandera` `DataFrameModel` with `pa.Field` definitions only and it catches data corruption and schema changes
   - this is the source of truth for what the columns are and what types they should be
   - this acts as the data dictionary
2. `behavioural_checks.py`
   - this contains the `@pa.check` and `@pa.dataframe_check` logic and statistical tests
   - these are defined as reusable functions and they check the business logic and anomaly detection using Z-scores, T-tests, and financial consistency
3. `schema.py`
   - this inherits from the base schema and adds the checks from `behavioural_checks.py`
   - this is the class that is actually imported for use in validating the raw credit card customer data

The following `pandera` functionality was used to write the validation checks in this schema

1. [`pandera`'s column checks](https://pandera.readthedocs.io/en/stable/dataframe_models.html#basic-usage) are used to perform column-level data validation
2. [`pandera`'s `DataFrame` Checks](https://pandera.readthedocs.io/en/stable/dataframe_models.html#dataframe-checks) are used to implement Dataset-level validation checks, anomaly detection checks and Z-score checks
3. [`pandera`'s hypothesis testing](https://pandera.readthedocs.io/en/stable/hypothesis.html)

This schema is now used to validate the raw data passed to us by the client

In [ ]:
%%time
try:
    validated_df = CreditCardCustomerSchema.validate(df)
except pa.errors.SchemaError as e:
    print(f"Validation failed: {str(e)}")

## Conclusion

This notebook as implemented a schema to validate incoming credit card customer data provided to us by the client. Here, we used it to validate the data we already had to validate customers for whom we knew if they churned or did not churn. Following the same approach, this schema can be used to validate new (unseen) data. By doing this, we ensure new customer data demonstrates the same behaviour as that of the data we used to identify at-risk customers. Importantly, we also ensure that our estimated business metrics (e.g. savings) from targeting at-risk customers, which we determined using the provide sample customer data, can also be realized by the client on new data.